# 01 - Audit dan Pembersihan Data

Sumber: `data/raw.csv`, satu baris = satu paket dari hub Bandung.

Tujuan tahap ini:
1. Menemukan masalah data dan menghitung jumlah barisnya.
2. Memperbaiki masalah yang **pasti** (duplikat, label, nilai mustahil).
3. Menandai masalah yang belum bisa diputuskan tanpa mengarang.

Prinsip:
- **Belum ada imputasi.** Imputer nanti di-fit hanya pada data train.
- Data test April-Juni 2026 belum disentuh.
- Semua perbaikan deterministik ada di `src/features.py` agar API memakai aturan yang sama.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Notebook bisa dijalankan dari folder notebooks/ atau dari akar repo.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features import clean_raw, KOTA_ALIAS, MAX_BERAT_KG, VOLUME_HUB_EKSTREM

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 160)

RAW_PATH = ROOT / "data" / "raw.csv"
CLEAN_PATH = ROOT / "data" / "clean.csv"

raw = pd.read_csv(RAW_PATH)
print("Ukuran data mentah:", raw.shape)
print("Periode:", raw["tanggal_pickup"].min(), "sampai", raw["tanggal_pickup"].max())
raw.head(3)

Ukuran data mentah: (40500, 13)
Periode: 2025-01-01 sampai 2026-06-30


,id_resi,tanggal_pickup,jam_pickup,kota_tujuan,jarak_km,jumlah_transit,layanan,janji_hari,berat_kg,metode_bayar,cuaca_saat_pickup,volume_hub_hari,terlambat
0,BDG69958899,2025-01-01,14,Tasikmalaya,109.0,2,HEMAT,4,5.93,COD,berawan,4990,0
1,BDG98718051,2025-01-01,15,Surabaya,678.0,2,HEMAT,5,0.78,non_COD,berawan,4769,0
2,BDG40074637,2025-01-01,14,Bekasi,128.0,1,REG,2,2.57,COD,cerah,4286,0


## 1. Duplikat

Cek duplikat baris penuh dan duplikat `id_resi` terpisah. Duplikat `id_resi`
berbahaya jika isinya berbeda; kalau isinya sama persis, cukup dibuang.

In [2]:
dup_baris = raw.duplicated()
dup_id = raw["id_resi"].duplicated(keep=False)
konflik = raw.loc[dup_id].groupby("id_resi").nunique().max(axis=1)

print("Duplikat identik          :", int(dup_baris.sum()))
print("id_resi unik              :", raw["id_resi"].nunique())
print("Baris dengan id_resi ganda:", int(dup_id.sum()))
print("Grup id_resi ganda        :", raw.loc[dup_id, "id_resi"].nunique())
print("Grup dengan isi berbeda   :", int((konflik > 1).sum()))

Duplikat identik          : 500
id_resi unik              : 40000
Baris dengan id_resi ganda: 1000
Grup id_resi ganda        : 500
Grup dengan isi berbeda   : 0


## 2. Nilai kategori

`layanan` punya variasi kapitalisasi dan singkatan. `kota_tujuan` punya
singkatan serta nama provinsi/kota yang sama. `metode_bayar` dan
`cuaca_saat_pickup` diperiksa untuk memastikan tidak ada variasi tersembunyi.

In [3]:
for col in ["layanan", "metode_bayar", "cuaca_saat_pickup"]:
    print(f"\n{col}:")
    print(raw[col].value_counts(dropna=False).to_string())

print("\nKota unik:", raw["kota_tujuan"].nunique())
print(raw["kota_tujuan"].value_counts().to_string())


layanan:
layanan
REG        23704
HEMAT       8001
EXPRESS     7983
Express      213
hemat        213
EXP          212
reg          174

metode_bayar:
metode_bayar
non_COD    28269
COD        12231

cuaca_saat_pickup:
cuaca_saat_pickup
cerah          16100
berawan        12016
hujan           8033
hujan_lebat     3945
NaN              406

Kota unik: 28
kota_tujuan
Jakarta          7697
Bekasi           3813
Surabaya         3358
Bogor            3256
Semarang         2786
Yogyakarta       2679
Tasikmalaya      2096
Cirebon          2021
Denpasar         1920
Medan            1907
Malang           1714
Makassar         1643
Palembang        1609
Balikpapan       1604
Pontianak        1249
jakarta           141
Jkt               137
bekasi            134
DKI Jakarta       122
Kota Bekasi       110
surabaya           99
Sby                90
MEDAN              76
DI Yogyakarta      60
Jogja              52
Bali               47
denpasar           45
Yogya              35


## 3. Nilai hilang dan nilai mustahil

Nilai hilang dibiarkan `NaN` pada tahap ini. Nilai mustahil seperti berat
negatif atau berat di atas batas wajar dijadikan `NaN`, bukan ditebak.

In [4]:
print("Jumlah nilai hilang per kolom:")
print(raw.isna().sum().to_string())

print("\nBerat <= 0 kg :", int((raw["berat_kg"] <= 0).sum()))
print("Berat >", MAX_BERAT_KG, "kg:", int((raw["berat_kg"] > MAX_BERAT_KG).sum()))
print("Jarak negatif :", int((raw["jarak_km"] < 0).sum()))
print("Jarak nol     :", int((raw["jarak_km"] == 0).sum()))
print("Jarak nol dengan transit > 0:", int(((raw["jarak_km"] == 0) & (raw["jumlah_transit"] > 0)).sum()))

print("\nKuartil volume_hub_hari:")
print(raw["volume_hub_hari"].quantile([0.5, 0.95, 0.99, 0.999, 1]).to_string())
print("Volume >", VOLUME_HUB_EKSTREM, ":", int((raw["volume_hub_hari"] > VOLUME_HUB_EKSTREM).sum()))

Jumlah nilai hilang per kolom:
id_resi                0
tanggal_pickup         0
jam_pickup             0
kota_tujuan            0
jarak_km             813
jumlah_transit         0
layanan                0
janji_hari             0
berat_kg               0
metode_bayar           0
cuaca_saat_pickup    406
volume_hub_hari        0
terlambat              0

Berat <= 0 kg : 81
Berat > 100.0 kg: 1205
Jarak negatif : 0
Jarak nol     : 0
Jarak nol dengan transit > 0: 0

Kuartil volume_hub_hari:
0.500     5358.000
0.950    10210.050
0.990    12828.000
0.999    15171.501
1.000    17979.000
Volume > 9000 : 2532


## 4. Tanggal dan jam

Tanggal harus bisa diparse dan berada di Januari 2025-Juni 2026.
Jam pickup harus 7-22. Keduanya juga divalidasi di API.

In [5]:
tanggal = pd.to_datetime(raw["tanggal_pickup"], errors="coerce")
print("Tanggal gagal diparse       :", int(tanggal.isna().sum()))
print("Tanggal di luar periode data:", int((~tanggal.between("2025-01-01", "2026-06-30")).sum()))
print("Jam di luar 7-22            :", int((~raw["jam_pickup"].between(7, 22)).sum()))

Tanggal gagal diparse       : 0
Tanggal di luar periode data: 0
Jam di luar 7-22            : 0


## 5. Terapkan pembersihan deterministik

`clean_raw` melakukan deduplikasi, normalisasi label, dan mengubah nilai
mustahil menjadi `NaN`. Imputasi sengaja tidak ada di fungsi ini.

In [6]:
clean = clean_raw(raw)

print("Mentah :", raw.shape)
print("Bersih :", clean.shape)
print("id_resi unik setelah bersih:", clean["id_resi"].nunique())

print("\nLabel layanan setelah bersih:")
print(clean["layanan"].value_counts().to_string())

print("\nKota setelah bersih:")
print(clean["kota_tujuan"].value_counts().to_string())

print("\nMetode bayar setelah bersih:")
print(clean["metode_bayar"].value_counts().to_string())

Mentah : (40500, 13)
Bersih : (40000, 22)
id_resi unik setelah bersih: 40000

Label layanan setelah bersih:
layanan
REG        23552
EXPRESS     8330
HEMAT       8118

Kota setelah bersih:
kota_tujuan
Jakarta        7996
Bekasi         4016
Surabaya       3501
Bogor          3211
Yogyakarta     2794
Semarang       2751
Tasikmalaya    2067
Cirebon        1994
Denpasar       1989
Medan          1957
Malang         1695
Makassar       1618
Balikpapan     1587
Palembang      1586
Pontianak      1238

Metode bayar setelah bersih:
metode_bayar
non_COD    27932
COD        12068


## 6. `janji_hari` versus layanan

Rentang SLA yang dianggap wajar: EXPRESS 1-3 hari, REG 2-4 hari,
HEMAT 4-6 hari. Baris di luar rentang **ditandai**, tidak diubah, karena
bisa saja ada kesepakatan khusus.

In [7]:
print("Sebaran janji_hari per layanan:")
print(pd.crosstab(clean["layanan"], clean["janji_hari"]).to_string())

print("\nBaris yang ditandai tidak sesuai:")
print(clean.loc[clean["flag_janji_tidak_sesuai"], "layanan"].value_counts().to_string())

Sebaran janji_hari per layanan:
janji_hari   1.0    2.0   3.0   4.0   5.0   6.0
layanan                                        
EXPRESS     3845   2856  1513    78    23    15
HEMAT         16     69    51  3877  2752  1353
REG           18  11336  8014  4172    10     2

Baris yang ditandai tidak sesuai:
layanan
HEMAT      136
EXPRESS    116
REG         30


## 7. Apakah nilai hilang berkaitan dengan target atau kota?

Jika nilai hilang jauh lebih sering terlambat, missingness bisa informatif.
Kalau hanya berbeda tipis, cukup ditandai dan diimputasi dari train.

In [8]:
for flag, nama in [
    ("flag_jarak_missing", "jarak_km hilang"),
    ("flag_cuaca_missing", "cuaca_saat_pickup hilang"),
]:
    tabel = clean.groupby(flag)["terlambat"].agg(["size", "mean"])
    print(f"\n{nama}:")
    print(tabel.round(4).to_string())

for col in ["jarak_km", "cuaca_saat_pickup"]:
    per_kota = clean.groupby("kota_tujuan").agg(
        n=(col, "size"), rasio_hilang=(col, lambda s: s.isna().mean())
    )
    per_kota = per_kota[per_kota["n"] >= 1000]["rasio_hilang"]
    print(f"\nRentang rasio {col} hilang per kota (n>=1000):")
    print(per_kota.agg(["min", "max"]).round(4).to_string())


jarak_km hilang:
                     size    mean
flag_jarak_missing               
False               39200  0.1785
True                  800  0.1588

cuaca_saat_pickup hilang:
                     size    mean
flag_cuaca_missing               
False               39600  0.1779
True                  400  0.1925

Rentang rasio jarak_km hilang per kota (n>=1000):
min    0.0140
max    0.0247

Rentang rasio cuaca_saat_pickup hilang per kota (n>=1000):
min    0.0065
max    0.0131


## 8. Tabel masalah data

Angka di bawah dihitung **setelah 500 duplikat identik dibuang**, supaya satu
masalah tidak terhitung dua kali. Angka mentah yang berbeda dicantumkan dalam
kurung agar mudah dicek ke `raw.csv`.

In [9]:
dup_n = int(raw.duplicated().sum())
raw_jarak_hilang = int(raw["jarak_km"].isna().sum())
raw_cuaca_hilang = int(raw["cuaca_saat_pickup"].isna().sum())
raw_berat_invalid = int(
    ((raw["berat_kg"] <= 0) | (raw["berat_kg"] > MAX_BERAT_KG)).sum()
)
raw_berat_neg = int((raw["berat_kg"] <= 0).sum())
raw_berat_besar = int((raw["berat_kg"] > MAX_BERAT_KG).sum())
raw_volume_ekstrem = int((raw["volume_hub_hari"] > VOLUME_HUB_EKSTREM).sum())
raw_layanan_alias = int((~raw["layanan"].isin(["EXPRESS", "REG", "HEMAT"])).sum())

kota_kunci = raw["kota_tujuan"].str.strip().str.lower()
kota_baku = kota_kunci.map(KOTA_ALIAS).fillna(raw["kota_tujuan"].str.strip().str.title())
raw_kota_alias = int(kota_baku.ne(raw["kota_tujuan"].str.strip()).sum())

layanan_alias = int(clean["flag_layanan_alias"].sum())
kota_alias = int(clean["flag_kota_alias"].sum())
jarak_hilang = int(clean["flag_jarak_missing"].sum())
cuaca_hilang = int(clean["flag_cuaca_missing"].sum())
berat_invalid = int(clean["flag_berat_invalid"].sum())
volume_ekstrem = int(clean["flag_volume_ekstrem"].sum())
janji_salah = int(clean["flag_janji_tidak_sesuai"].sum())

rasio_jarak_hilang = clean.loc[clean["flag_jarak_missing"], "terlambat"].mean()
rasio_jarak_ada = clean.loc[~clean["flag_jarak_missing"], "terlambat"].mean()
rasio_cuaca_hilang = clean.loc[clean["flag_cuaca_missing"], "terlambat"].mean()
rasio_cuaca_ada = clean.loc[~clean["flag_cuaca_missing"], "terlambat"].mean()


def rentang_hilang(col):
    per_kota = clean.groupby("kota_tujuan").agg(
        n=(col, "size"), rasio=(col, lambda s: s.isna().mean())
    )
    nilai = per_kota.loc[per_kota["n"] >= 1000, "rasio"]
    return f"{nilai.min():.1%}-{nilai.max():.1%}"


baris_missing = (
    f"jarak {rasio_jarak_hilang:.1%} vs {rasio_jarak_ada:.1%}; "
    f"cuaca {rasio_cuaca_hilang:.1%} vs {rasio_cuaca_ada:.1%}; "
    f"kota {rentang_hilang('jarak_km')} / {rentang_hilang('cuaca_saat_pickup')}"
)

tabel_masalah = pd.DataFrame(
    [
        ["Duplikat identik", f"{dup_n}", "Hapus dengan drop_duplicates",
         "Isi sama persis; jika dibiarkan, bobot baris menjadi ganda."],
        ["id_resi ganda", "1.000 baris (500 grup)", "Terhapus bersama duplikat identik",
         "Tidak ada konflik nilai; setelah bersih tersisa 40.000 id unik."],
        ["Label layanan tidak baku", f"{layanan_alias} ({raw_layanan_alias} mentah)",
         "Petakan ke EXPRESS/REG/HEMAT",
         "Hanya beda kapitalisasi atau singkatan (Express, EXP, hemat, reg)."],
        ["Ejaan kota tidak baku", f"{kota_alias} ({raw_kota_alias} mentah)",
         "Petakan dengan daftar alias",
         "28 ejaan menjadi 15 kota; Bali hanya muncul sebagai Denpasar di data."],
        ["jarak_km kosong", f"{jarak_hilang} ({raw_jarak_hilang} mentah)",
         "Biarkan NaN, isi flag_jarak_missing",
         "Imputasi median per kota nanti di-fit pada train saja agar tidak bocor."],
        ["cuaca_saat_pickup kosong", f"{cuaca_hilang} ({raw_cuaca_hilang} mentah)",
         "Biarkan NaN, isi flag_cuaca_missing",
         "Cuaca tidak diketahui sendiri bisa bermakna; imputasi menunggu train."],
        ["berat_kg tidak wajar", f"{berat_invalid} ({raw_berat_invalid} mentah)",
         "Jadikan NaN, isi flag_berat_invalid",
         f"{raw_berat_neg} negatif dan {raw_berat_besar} >{MAX_BERAT_KG:g} kg; "
         "kemungkinan salah satuan gram, tetapi tidak dikoreksi karena ambigu."],
        ["volume_hub_hari ekstrem",
         f"{volume_ekstrem} ({raw_volume_ekstrem} mentah, >{VOLUME_HUB_EKSTREM})",
         "Tandai dengan flag_volume_ekstrem",
         "Bisa lonjakan nyata; jangan dibuang tanpa bukti anomali sistem."],
        ["janji_hari tidak sesuai layanan", f"{janji_salah}",
         "Tandai dengan flag_janji_tidak_sesuai",
         "Rentang wajar EXPRESS 1-3, REG 2-4, HEMAT 4-6; bisa ada SLA khusus."],
        ["jam_pickup di luar 7-22", "0", "Tidak ada yang diubah",
         "Seluruh jam valid; API tetap memvalidasi rentang ini."],
        ["Tanggal tidak valid / di luar periode", "0", "Tidak ada yang diubah",
         "Seluruh tanggal valid dan berada di Januari 2025-Juni 2026."],
        ["jarak_km nol / negatif", "0", "Tidak ada yang diubah",
         "Jarak minimum 90 km dan tidak ada nilai nol."],
        ["Nilai negatif selain berat", "0", "Tidak ada yang diubah",
         "jumlah_transit, janji_hari, dan volume_hub_hari tidak negatif."],
        ["Spasi berlebih pada kategori", "0", "strip() tetap dipakai",
         "Tidak ditemukan spasi di awal/akhir; normalisasi tetap untuk keamanan API."],
        ["Nilai hilang terkait target", baris_missing, "Tidak ada penanganan khusus",
         "Selisih tingkat terlambat kecil dan tidak konsisten; missingness tidak kuat."],
    ],
    columns=["Masalah", "Baris", "Penanganan", "Alasan"],
)

tabel_masalah

,Masalah,Baris,Penanganan,Alasan
0,Duplikat identik,500,Hapus dengan drop_duplicates,"Isi sama persis; jika dibiarkan, bobot baris menjadi ganda."
1,id_resi ganda,1.000 baris (500 grup),Terhapus bersama duplikat identik,Tidak ada konflik nilai; setelah bersih tersisa 40.000 id unik.
2,Label layanan tidak baku,589 (812 mentah),Petakan ke EXPRESS/REG/HEMAT,"Hanya beda kapitalisasi atau singkatan (Express, EXP, hemat, reg)."
3,Ejaan kota tidak baku,1132 (1148 mentah),Petakan dengan daftar alias,28 ejaan menjadi 15 kota; Bali hanya muncul sebagai Denpasar di data.
4,jarak_km kosong,800 (813 mentah),"Biarkan NaN, isi flag_jarak_missing",Imputasi median per kota nanti di-fit pada train saja agar tidak bocor.
5,cuaca_saat_pickup kosong,400 (406 mentah),"Biarkan NaN, isi flag_cuaca_missing",Cuaca tidak diketahui sendiri bisa bermakna; imputasi menunggu train.
6,berat_kg tidak wajar,1274 (1286 mentah),"Jadikan NaN, isi flag_berat_invalid","81 negatif dan 1205 >100 kg; kemungkinan salah satuan gram, tetapi tidak dikoreksi karena ambigu."
7,volume_hub_hari ekstrem,"2502 (2532 mentah, >9000)",Tandai dengan flag_volume_ekstrem,Bisa lonjakan nyata; jangan dibuang tanpa bukti anomali sistem.
8,janji_hari tidak sesuai layanan,282,Tandai dengan flag_janji_tidak_sesuai,"Rentang wajar EXPRESS 1-3, REG 2-4, HEMAT 4-6; bisa ada SLA khusus."
9,jam_pickup di luar 7-22,0,Tidak ada yang diubah,Seluruh jam valid; API tetap memvalidasi rentang ini.


## 9. Simpan data bersih dan cek akhir

`clean.csv` masih berisi `NaN` dan kolom `flag_*`. Imputasi dan encoding
dilakukan pada tahap berikutnya, setelah split waktu.

In [10]:
clean.to_csv(CLEAN_PATH, index=False)

# Cek cepat agar hasil tidak diam-diam rusak.
assert clean["id_resi"].is_unique
assert not clean.duplicated().any()
assert set(clean["layanan"]) == {"EXPRESS", "REG", "HEMAT"}
assert set(clean["metode_bayar"]) <= {"COD", "non_COD"}
assert clean["berat_kg"].dropna().between(0, MAX_BERAT_KG).all()
assert clean["flag_tanggal_invalid"].sum() == 0
assert clean["flag_jam_invalid"].sum() == 0

print("Disimpan ke:", CLEAN_PATH)
print("Ukuran akhir:", clean.shape)
print("Rasio terlambat:", round(clean["terlambat"].mean(), 4))
print("Kolom:", list(clean.columns))

Disimpan ke: C:\PROJECTVibeCoding\ModelMLStudyCase8\data\clean.csv
Ukuran akhir: (40000, 22)
Rasio terlambat: 0.1781
Kolom: ['id_resi', 'tanggal_pickup', 'jam_pickup', 'kota_tujuan', 'jarak_km', 'jumlah_transit', 'layanan', 'janji_hari', 'berat_kg', 'metode_bayar', 'cuaca_saat_pickup', 'volume_hub_hari', 'terlambat', 'flag_layanan_alias', 'flag_kota_alias', 'flag_tanggal_invalid', 'flag_jam_invalid', 'flag_berat_invalid', 'flag_jarak_missing', 'flag_cuaca_missing', 'flag_volume_ekstrem', 'flag_janji_tidak_sesuai']


## Catatan untuk tahap berikutnya

- Split waktu: train sampai 31 Desember 2025, validation Januari-Maret 2026,
  test April-Juni 2026. Test belum dibuka.
- Imputer, scaler, dan encoder hanya di-fit pada train.
- `flag_*` ikut menjadi fitur agar model tahu kapan nilai diimputasi.